In [1]:
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import networkx as nx
from utilities import preprocess

In [ ]:
filename='processed_data\itigi_comments_copnsolidated.parqet'
litigi=pd.read_parquet(filename).drop_duplicates(subset=['author','body','id', 'created_utc'])
litigi['created_dt']=(pd.to_datetime(litigi['created_utc'],unit='s'))
litigi['body_preprocessed']=litigi['body'].apply(lambda x: preprocess(x))

litigi.head()

In [ ]:
litigi.created_dt.max()

In [ ]:
litigi.describe()

In [ ]:
litigi_subsample=litigi[:1000][['author','body_preprocessed','id', 'is_submitter', 'link_id', 'parent_id','created_dt']]
litigi_subsample

In [ ]:
def create_nx_graph(df:pd.DataFrame):
    G=nx.DiGraph()
    G.add_nodes_from([(id,{'type':'comment'}) for id in df['id']])
    G.add_nodes_from([(link_id,{'type':'submission'}) for link_id in df['link_id'].drop_duplicates().str.split('_').str[1]])
    G.add_edges_from([a for a in zip(df['id'],df['parent_id'].drop_duplicates().str.split('_').str[1])])
    return G

G=create_nx_graph(litigi_subsample)
nx.node_link_data(G)

For a chord diagram look into: https://plotly.com/python/v3/chord-diagram/

In [ ]:
from analysis import generate_graph_figure
pos = nx.spring_layout(G)
nx.set_node_attributes(G, pos, "pos")
generate_graph_figure(G)

In [ ]:
G.edges

In [ ]:
G['dlp0rm3']

In [ ]:
pos = nx.spring_layout(G)
nx.draw(G, pos ,  width=0.4, 
        node_color='lightblue', node_size=10)

In [ ]:
df_pivot = pd.pivot_table(
    litigi,
    index='author',
    columns="sentiment",
    aggfunc='size'
)

In [ ]:
df_pivot=df_pivot.fillna(0)
df_pivot=df_pivot.sort_values('positive',ascending=False)
df_pivot['positive_ratio']=df_pivot['positive']/(df_pivot['negative']+df_pivot['neutral'])
df_pivot['p/n']=df_pivot['positive']/(df_pivot['negative'])

In [ ]:
df_pivot

In [ ]:
litigi[(litigi['num_words']==1) & (litigi['body_preprocessed']!='[deleted]')][['body_preprocessed']]

In [ ]:
df_pivot.to_csv('/content/drive/MyDrive/AI/data/output/litigi_sentiment.csv')

In [ ]:
litigi[(litigi['body_preprocessed']!='[deleted]') &(litigi['body_preprocessed']!='[removed]') ]['num_words'].hist(bins=range(75),grid=False)

In [ ]:
sns.kdeplot(litigi[ (litigi['num_words']<100) &(litigi['num_words']>1) ]['num_words'])
plt.show()

In [ ]:
author_list=['tommyrugby','sda_express','SpiegoLeDiscussioni','PHEELZ']

In [ ]:
df_filtered = litigi[litigi['author'].isin(author_list)]

In [ ]:
for author in author_list:
    plt.hist(df_filtered[df_filtered['author'] == author]['num_words'], alpha=0.5, label=author,bins=range(75))
plt.legend(loc='upper right')
plt.xlabel('Number of words')
plt.ylabel('Frequency')
plt.show()

In [ ]:

histograms = []
for author in author_list:
    author_hist, _ = np.histogram(df_filtered[df_filtered['author'] == author]['num_words'],bins=range(50))
    histograms.append(author_hist)

# Plot histograms side by side
plt.figure(figsize=(19, 10))
width = 0.2
x = np.arange(len(histograms[0]))

for i, author_hist in enumerate(histograms):
    plt.bar(x + i*width, author_hist, width=width, alpha=0.9, label=author_list[i])

plt.xlabel('Number of words')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()


In [ ]:
date=pd.to_datetime(litigi['created_utc'], unit='s').dt.date
date

In [ ]:
plt.hist(date,bins=50,alpha=0.5)

plt.show()

In [ ]:
time=pd.to_datetime(litigi['created_utc'], unit='s').dt.hour
time

In [ ]:
plt.hist(time,alpha=0.9,rwidth=0.7,bins=24)

plt.show()

In [ ]:
histograms = []
for author in author_list:
    author_hist, _ = np.histogram(pd.to_datetime(df_filtered[df_filtered['author'] == author]['created_utc'], unit='s').dt.hour,bins=24,density=True)
    histograms.append(author_hist)

# Plot histograms side by side
plt.figure(figsize=(19, 10))
width = 0.2
x = np.arange(len(histograms[0]))

for i, author_hist in enumerate(histograms):
    plt.bar(x + i*width, author_hist, width=width, alpha=0.9, label=author_list[i])

plt.xlabel('hour')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()

In [ ]:
df_filtered = litigi[litigi['author'].isin(author_list)]

In [ ]:
plt.figure(figsize=(19, 10))

for author in author_list:
    sns.kdeplot(pd.to_datetime(df_filtered[df_filtered['author'] == author]['created_utc'], unit='s').dt.hour,label=author)
    #histograms.append(author_hist)

plt.xlabel('hour')
plt.ylabel('Frequency')
plt.legend(loc='upper right')
plt.show()

In [ ]:
plt.figure(figsize=(19, 10))

sns.kdeplot(data=df_filtered,x=pd.to_datetime(df_filtered['created_utc'], unit='s').dt.hour,label=author,hue='author', common_norm=False)
plt.show()